# Data Science Challenge – Town A (Seasonal / H3N2 Flu Vaccine) · Group B
**GitHub:** https://github.com/UH-RMDS-DataChallenge-2026 · **Target:** `seasonal_vaccine` (1 = will accept the vaccine)

**The problem.** The mayor must decide *who* receives a vaccine. Everyone will be infected (100 % infection). Each vaccine costs **$300** (also paid in cash to anyone who wants one but is not sent one) and each hospitalisation costs **$18,500**. Our goal is to **minimise the town's total cost**, not to maximise accuracy.

| Section | Content |
|---|---|
| 0 | Setup and configuration |
| 1 | Data loading and integrity checks |
| 2 | Exploratory Data Analysis (EDA) |
| 3 | The task-specific cost metric |
| 4 | Feature engineering and pre-processing |
| 5 | Pre-processing experiments |
| 6 | Model comparison (5-fold cross-validation) |
| 7 | Hyper-parameter tuning |
| 8 | Results, ranking and diagnostics |
| 9 | Sensitivity analysis of our assumptions |
| 10 | Submission files (top 5) |
| 11 | Explainable AI (XAI) |
| 12 | Summary for non-experts |

*This notebook merges the work of the whole team (EDA, baseline models, XGBoost/LightGBM and feature engineering, H3N2-specific vaccine effectiveness, holdout check and cost-based XAI, cost metric and sensitivity analysis).*

## 0. Setup and configuration

In [ ]:
# Install gradient-boosting libraries and SHAP if they are missing (Colab usually has them)
%pip install -q xgboost lightgbm shap

In [ ]:
# ---- Core libraries ----
import warnings, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# ---- scikit-learn: pipelines, pre-processing, models, metrics ----
from sklearn.model_selection import StratifiedKFold, RandomizedSearchCV, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, HistGradientBoostingClassifier
from sklearn.base import clone
from sklearn.metrics import (roc_auc_score, log_loss, accuracy_score, brier_score_loss,
                             precision_score, recall_score, f1_score,
                             ConfusionMatrixDisplay, RocCurveDisplay)
from sklearn.calibration import calibration_curve
from sklearn.inspection import permutation_importance, PartialDependenceDisplay
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

warnings.filterwarnings("ignore")                  # keep the output readable
sns.set_theme(style="whitegrid")                   # consistent plot style
pd.set_option("display.max_columns", 100)          # show all columns in tables

RANDOM_STATE = 42                                  # one seed used everywhere -> reproducible results
np.random.seed(RANDOM_STATE); random.seed(RANDOM_STATE)

In [ ]:
# ---- Data location: our GitHub repository (Raw files) ----
BASE = "https://raw.githubusercontent.com/UH-RMDS-DataChallenge-2026/data-science-challenge/main/"
TRAIN_FILE, TEST_FILE, SAMPLE_FILE = ("dataset_A_training.csv", "dataset_A_testing.csv",
                                      "dataset_A_example_submission.csv")

GROUP_NUMBER = "B"                     # our group ID, used in the submission file names
TARGET, ID_COL = "seasonal_vaccine", "respondent_id"

# ---- Town A costs (hackathon brief) ----
VACCINE_COST  = 300                    # $ per vaccine, also paid in cash if a vaccine is wanted but not sent
HOSPITAL_COST = 18_500                 # $ per hospitalisation (H3N2 flu)

# ---- Evaluation settings ----
N_FOLDS = 5                            # folds for cross-validation
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)   # same folds for every model